# SWAP-Constrained Unitary Compilation — Evaluation

Evaluates the model trained with `; swap_count=N` prompt conditioning against the
baseline model (no SWAP count in prompt).

**What we measure:**

1. **Compilation accuracy per requested SWAP count** — Does requesting fewer SWAPs hurt the model's ability to compile the target unitary?
2. **Exact-count compliance** — Does the model actually produce circuits with the requested number of SWAPs?
   - `exact_compliance_rate = fraction of circuits where actual_swap_count == requested`
   - `mean_swap_count` per requested count (should track the diagonal)
3. **Comparison to baseline** — Does the SWAP-conditioned model retain the same compilation quality when given a high/unconstrained count?

Note: the model is trained on exact counts (`swap_count=N`), not upper bounds.
To minimise SWAPs on hardware, request `swap_count=0`.

Run `swap_gate_analysis.ipynb` first to understand the status quo.

## Setup

In [ ]:
import ast
import random
import sys
from pathlib import Path

import hydra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hydra.core.global_hydra import GlobalHydra
from IPython.display import display
from tqdm.auto import tqdm

sys.path.append("/home/a-ldungl/projects/qcircuit-generation")

from notebooks.shared.bootstrap import setup_notebook_paths
PROJECT_ROOT = setup_notebook_paths()

from notebooks.shared.evaluation_artifacts import make_artifact_dir, save_dataframe, save_figure, save_json, save_pickle
from my_genQC.inference.sampling import generate_compilation_tensors, decode_tensors_to_backend
from my_genQC.inference.evaluation_helper import get_unitaries
from my_genQC.inference.eval_metrics import UnitaryInfidelityNorm, UnitaryFrobeniusNorm
from my_genQC.pipeline.diffusion_pipeline import DiffusionPipeline
from my_genQC.platform.simulation import CircuitBackendType, Simulator
from my_genQC.platform.tokenizer.circuits_tokenizer import CircuitTokenizer
from my_genQC.utils.misc_utils import infer_torch_device
from quantum_diffusion.data.dataset import DatasetLoader

In [ ]:
# -- Configuration (edit as needed) -------------------------------------------

# Eval dataset — same one used for baseline evaluation
EVAL_DATASET_PATH = "./artifacts/datasets/unitary-baseline-reproduction/eval/qiskit"

# SWAP-constrained model (trained on swap-constrained dataset)
SWAP_MODEL_DIR = "./artifacts/models/unitary-swap-constrained/unitary_swap_constrained"

# Baseline model (no SWAP count conditioning)
BASELINE_MODEL_DIR = "./artifacts/models/unitary-baseline-reproduction/paper_unitary"

# Exact SWAP counts to request at inference (model was trained on exact counts)
SWAP_COUNTS = [0, 1, 2, 3]
# High count used as the 'unconstrained' reference for the constrained model
UNCONSTRAINED_COUNT = 6

# Evaluation settings
NUM_EVAL_UNITARIES = 128
SAMPLES_PER_UNITARY = 64
GUIDANCE_SCALE = 7.5
SAMPLE_STEPS = 20
MAX_GATES = 12
AUTO_BATCH_SIZE = 128
EXACT_DISTANCE_TOL = 1e-8
SEED = 42

ARTIFACT_SUBDIR = "unitary-swap-constrained"
RUN_NAME = "swap_count_evaluation"

device = str(infer_torch_device())
print(f"Device: {device}")

ARTIFACT_DIR = make_artifact_dir(PROJECT_ROOT, ARTIFACT_SUBDIR, RUN_NAME)
print(f"Artifacts: {ARTIFACT_DIR}")

## Load models and dataset

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_pipeline(model_dir):
    path = Path(model_dir).expanduser().resolve()
    pipeline = DiffusionPipeline.from_config_file(config_path=str(path) + "/", device=device)
    pipeline.guidance_sample_mode = "rescaled"
    pipeline.scheduler.set_timesteps(SAMPLE_STEPS)
    return pipeline


set_seed(SEED)

# Eval dataset
GlobalHydra.instance().clear()
with hydra.initialize(version_base=None, config_path="../../../conf"):
    cfg = hydra.compose(config_name="config.yaml", overrides=["evaluation=paper_srv"])
eval_cfg = cfg["evaluation"]
eval_cfg.dataset = str(Path(EVAL_DATASET_PATH).expanduser().resolve())
eval_cfg.model_dir = str(Path(SWAP_MODEL_DIR).expanduser().resolve())
eval_cfg.save_output = False
eval_cfg.wandb.enable = False

loader = DatasetLoader(eval_cfg, device=device)
eval_dataset = loader.load_dataset(eval_cfg.dataset, load_embedder=False)
gate_pool = eval_dataset.gate_pool
print(f"Eval dataset: {len(eval_dataset.y):,} circuits, gate_pool: {gate_pool}")

# Shared tokenizer and simulator
vocabulary = {gate: idx for idx, gate in enumerate(gate_pool)}
tokenizer = CircuitTokenizer(vocabulary)
simulator = Simulator(CircuitBackendType.QISKIT)

# Models
print("\nLoading SWAP-constrained model...")
swap_pipeline = load_pipeline(SWAP_MODEL_DIR)

print("Loading baseline model...")
baseline_pipeline = load_pipeline(BASELINE_MODEL_DIR)

In [ ]:
# Select the same target unitaries for all experiments
rng = random.Random(SEED)
eval_indices = rng.sample(range(len(eval_dataset.y)), k=min(NUM_EVAL_UNITARIES, len(eval_dataset.y)))
print(f"Selected {len(eval_indices)} target unitaries")

## Evaluation loop

In [ ]:
def make_prompt(gate_pool, swap_count=None):
    """Build the text prompt, optionally appending an exact SWAP count."""
    base = f"Compile using: {[str(g) for g in gate_pool]}"
    if swap_count is not None:
        return f"{base}; swap_count={swap_count}"
    return base


def evaluate(pipeline, model_label: str, swap_count):  # swap_count: int or None
    """Run evaluation for one (model, requested SWAP count) combination."""
    prompt = make_prompt(gate_pool, swap_count)
    rows = []

    for idx in tqdm(eval_indices, desc=f"{model_label} | swap_count={swap_count}", leave=False):
        target_u_split = eval_dataset.U[idx].cpu()
        target_u = target_u_split[0].numpy() + 1j * target_u_split[1].numpy()

        tensors_out = generate_compilation_tensors(
            pipeline=pipeline,
            prompt=[prompt],
            U=target_u_split.unsqueeze(0).to(device),
            samples=SAMPLES_PER_UNITARY,
            system_size=eval_dataset.x.shape[1],
            num_of_qubits=getattr(eval_dataset.params_config, "num_of_qubits", eval_dataset.x.shape[1]),
            max_gates=MAX_GATES,
            g=GUIDANCE_SCALE,
            auto_batch_size=AUTO_BATCH_SIZE,
            enable_params=False,
            no_bar=True,
        )

        decoded_circuits, _ = decode_tensors_to_backend(
            simulator=simulator,
            tokenizer=tokenizer,
            tensors=tensors_out,
            params=None,
            silent=True,
            n_jobs=1,
            filter_errs=False,
        )

        for qc in decoded_circuits:
            if qc is None:
                rows.append({"dataset_idx": idx, "actual_swap": None, "exact": False,
                             "valid": False, "count_matched": None})
                continue
            actual_swap = sum(1 for inst in qc.data if inst.operation.name == "swap")
            gen_u = simulator.backend.get_unitary(qc)
            dist = float(UnitaryInfidelityNorm.distance(
                torch.as_tensor(gen_u).to(torch.complex64),
                torch.as_tensor(target_u).to(torch.complex64),
            ).item())
            exact = dist <= EXACT_DISTANCE_TOL
            count_matched = (actual_swap == swap_count) if swap_count is not None else None
            rows.append({
                "dataset_idx": idx,
                "actual_swap": actual_swap,
                "exact": exact,
                "valid": True,
                "count_matched": count_matched,
            })

    df = pd.DataFrame(rows)
    valid = df[df["valid"]]
    exact_found_rate = valid.groupby("dataset_idx")["exact"].any().mean() if len(valid) else float("nan")
    exact_compliance = float(df["count_matched"].dropna().mean()) if df["count_matched"].notna().any() else float("nan")
    mean_abs_err = float((df["actual_swap"].dropna() - swap_count).abs().mean()) if swap_count is not None and len(df["actual_swap"].dropna()) else float("nan")

    summary = {
        "model": model_label,
        "requested_swap_count": swap_count,
        "prompt": prompt,
        "exact_found_rate": exact_found_rate,
        "mean_exact_count": float(valid.groupby("dataset_idx")["exact"].sum().mean()) if len(valid) else float("nan"),
        "valid_decode_rate": float(df["valid"].mean()),
        "mean_actual_swap": float(valid["actual_swap"].mean()) if len(valid) else float("nan"),
        "median_actual_swap": float(valid["actual_swap"].median()) if len(valid) else float("nan"),
        "zero_swap_frac": float((valid["actual_swap"] == 0).mean()) if len(valid) else float("nan"),
        "exact_compliance_rate": exact_compliance,
        "mean_abs_swap_error": mean_abs_err,
    }
    print(
        f"{model_label} | requested={swap_count}: "
        f"exact_found={exact_found_rate:.3f}, "
        f"mean_actual_swap={summary['mean_actual_swap']:.2f}, "
        f"compliance={exact_compliance:.3f}"
    )
    return df, summary

In [ ]:
set_seed(SEED)

results = {}  # (model_label, requested_swap_count) -> {"df": ..., "summary": ...}

# --- SWAP-constrained model: evaluate across all counts + unconstrained reference ---
for count in SWAP_COUNTS + [UNCONSTRAINED_COUNT]:
    df, summary = evaluate(swap_pipeline, "swap_constrained", count)
    results[("swap_constrained", count)] = {"df": df, "summary": summary}

# --- Baseline model: no swap_count in prompt (as it was trained) ---
df_base, summary_base = evaluate(baseline_pipeline, "baseline", swap_count=None)
results[("baseline", None)] = {"df": df_base, "summary": summary_base}

print("\nDone.")

## Results

In [ ]:
summaries = [v["summary"] for v in results.values()]
summary_df = pd.DataFrame(summaries)
display(summary_df.round(4))

save_dataframe(summary_df, ARTIFACT_DIR / "swap_budget_summary.csv", index=False)

In [ ]:
# Plot 1: Compilation accuracy and mean SWAP count vs. requested count
swap_rows = [v["summary"] for (m, b), v in results.items() if m == "swap_constrained"]
swap_rows.sort(key=lambda r: r["requested_swap_count"])
counts = [r["requested_swap_count"] for r in swap_rows]
exact_rates = [r["exact_found_rate"] for r in swap_rows]
mean_actuals = [r["mean_actual_swap"] for r in swap_rows]

baseline_exact = results[("baseline", None)]["summary"]["exact_found_rate"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=130)

# Accuracy
axes[0].plot(counts, exact_rates, marker="o", color="#5f8fcb", linewidth=2, label="swap_constrained")
axes[0].axhline(baseline_exact, color="#d85c41", linestyle="--", linewidth=1.5, label=f"baseline ({baseline_exact:.3f})")
axes[0].set_xlabel("Requested swap_count")
axes[0].set_ylabel("Exact-found rate")
axes[0].set_title("Compilation accuracy vs. requested SWAP count")
axes[0].legend()
axes[0].grid(True, alpha=0.25)
axes[0].set_xticks(counts)

# Actual mean SWAP vs. requested (perfect compliance = diagonal)
axes[1].plot(counts, mean_actuals, marker="s", color="#63b17d", linewidth=2, label="mean actual SWAP")
axes[1].plot(counts, counts, "k--", linewidth=1, alpha=0.5, label="requested (perfect)")
axes[1].set_xlabel("Requested swap_count")
axes[1].set_ylabel("Mean actual SWAP count")
axes[1].set_title("SWAP count compliance (actual vs. requested)")
axes[1].legend()
axes[1].grid(True, alpha=0.25)
axes[1].set_xticks(counts)

plt.tight_layout()
save_figure(fig, ARTIFACT_DIR / "accuracy_vs_swap_count.png")
plt.show()

In [ ]:
# Plot 2: Actual SWAP distribution per requested count
fig, axes = plt.subplots(1, len(SWAP_COUNTS), figsize=(4 * len(SWAP_COUNTS), 4), dpi=130, sharey=True)

max_swap_seen = max(
    results[("swap_constrained", c)]["df"]["actual_swap"].dropna().max()
    for c in SWAP_COUNTS
    if len(results[("swap_constrained", c)]["df"]["actual_swap"].dropna()) > 0
)
bins = np.arange(-0.5, max_swap_seen + 1.5, 1)

for ax, count in zip(axes, SWAP_COUNTS):
    df_c = results[("swap_constrained", count)]["df"]
    valid_swaps = df_c[df_c["valid"]]["actual_swap"].dropna().astype(int)
    ax.hist(valid_swaps, bins=bins, color="#5f8fcb", edgecolor="white")
    ax.axvline(count, color="#d85c41", linestyle="--", linewidth=1.5)
    compliance = (valid_swaps == count).mean()
    ax.set_title(f"swap_count={count}\ncompliance {compliance*100:.1f}%", fontsize=10)
    ax.set_xlabel("Actual SWAP count")
    if ax is axes[0]:
        ax.set_ylabel("Count")
    ax.grid(True, alpha=0.2, axis="y")

fig.suptitle("Actual SWAP Distribution per Requested Count (swap_constrained model)", fontsize=12, y=1.01)
plt.tight_layout()
save_figure(fig, ARTIFACT_DIR / "swap_distribution_per_count.png")
plt.show()

In [ ]:
# Heatmap: requested count vs. actual SWAP count
all_counts = SWAP_COUNTS + [UNCONSTRAINED_COUNT]
max_actual = int(max(
    results[("swap_constrained", c)]["df"]["actual_swap"].dropna().max()
    for c in all_counts
    if len(results[("swap_constrained", c)]["df"]["actual_swap"].dropna()) > 0
))

heatmap = np.zeros((len(all_counts), max_actual + 1))
for i, count in enumerate(all_counts):
    valid_swaps = results[("swap_constrained", count)]["df"]
    valid_swaps = valid_swaps[valid_swaps["valid"]]["actual_swap"].dropna().astype(int)
    for k in range(max_actual + 1):
        heatmap[i, k] = (valid_swaps == k).sum()
    row_sum = heatmap[i].sum()
    if row_sum > 0:
        heatmap[i] /= row_sum

fig, ax = plt.subplots(figsize=(max_actual + 2, len(all_counts) + 1), dpi=120)
im = ax.imshow(heatmap, aspect="auto", cmap="Blues", vmin=0)
ax.set_xticks(range(max_actual + 1))
ax.set_xticklabels(range(max_actual + 1))
ax.set_yticks(range(len(all_counts)))
ax.set_yticklabels([f"swap_count={c}" for c in all_counts])
ax.set_xlabel("Actual SWAP count")
ax.set_ylabel("Requested swap_count")
ax.set_title("Fraction of circuits at each actual SWAP count\n(swap_constrained model) — diagonal = perfect compliance")
plt.colorbar(im, ax=ax, label="Fraction")

for i in range(len(all_counts)):
    for j in range(max_actual + 1):
        ax.text(j, i, f"{heatmap[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if heatmap[i, j] > 0.5 else "black")

plt.tight_layout()
save_figure(fig, ARTIFACT_DIR / "requested_vs_actual_swap_heatmap.png")
plt.show()

In [ ]:
# Save all raw results
for (model, budget), payload in results.items():
    label = f"{model}_budget{budget}"
    save_dataframe(payload["df"], ARTIFACT_DIR / f"{label}_per_circuit.csv", index=False)

save_json(
    {
        "eval_dataset_path": EVAL_DATASET_PATH,
        "swap_model_dir": SWAP_MODEL_DIR,
        "baseline_model_dir": BASELINE_MODEL_DIR,
        "swap_budgets": SWAP_BUDGETS,
        "unconstrained_budget": UNCONSTRAINED_BUDGET,
        "num_eval_unitaries": NUM_EVAL_UNITARIES,
        "samples_per_unitary": SAMPLES_PER_UNITARY,
        "guidance_scale": GUIDANCE_SCALE,
        "exact_distance_tol": EXACT_DISTANCE_TOL,
        "seed": SEED,
        "summaries": summaries,
    },
    ARTIFACT_DIR / "run_config.json",
)
print(f"Saved evaluation artifacts to {ARTIFACT_DIR}")

## Interpretation Guide

**The heatmap is the key diagnostic.** A well-trained model shows mass concentrated on the diagonal: row `swap_count=0` mostly at column 0, row `swap_count=2` mostly at column 2, etc. Blurring across adjacent columns is expected (CLIP number separation is ~0.30 L2 for integers 1–3); complete lack of diagonal structure means the conditioning isn't working.

**Exact-count compliance per requested value:**
- `swap_count=0`: cleanest signal (largest CLIP step from 0→1 at L2=0.52). Compliance > 60% is a good result.
- `swap_count=1/2/3`: CLIP separation is weaker (~0.30). Expect softer compliance; mean_actual_swap tracking the direction matters more than hitting exactly.

**Accuracy vs. requested count:**
- Mild accuracy drop at `swap_count=0` is expected — some unitaries genuinely need SWAPs for correct compilation.
- At `swap_count=2+`, accuracy should be within a few percent of the baseline. A large drop suggests the added token is hurting general compilation quality.

**Unconstrained reference (`swap_count=6`) vs. baseline:**
- These should have comparable `exact_found_rate`. A large gap means the extra token in the prompt shifts the distribution even when the count is loose — a sign the model needs more training.

**If results are disappointing:**
- Compliance weak across all counts → consider a dedicated numerical embedding (replace CLIP number tokens with a learned 1D lookup projected into the 512-dim condition space)
- Compliance OK for 0 but not 1/2/3 → CLIP number separation is the bottleneck; categorical labels or numerical embedding would help
- Accuracy drops significantly even at high counts → balance the dataset by swap_count bucket and/or train longer